### Refund Agent — Eval Dataset Setup

Builds a curated evaluation dataset for the Refund Agent and registers it
as a UC-managed MLflow dataset at
`{CATALOG}.evaluations.refund_agent_eval_dataset`.

**Why a separate task:** matches the pattern in
`stages/operational_evaluation.ipynb` for the Operational
Supervisor — dataset creation is a one-time, governed, unconditional step;
evaluation is the ongoing, opt-in step. Splitting them means:

- The dataset is visible in Catalog Explorer after every deploy, even when
  `SKIP_EVAL=true` (the default for `Refund_Evaluation`).
- Re-running the eval task doesn't rebuild the dataset — same records
  every time → reproducible eval runs.
- A future MemAlign / `optimize_prompts()` job can consume the same UC
  dataset without re-deriving it.

This task runs after `Refund_Recommender_Agent` (so the streaming pipeline
has had time to land delivered orders into `{CATALOG}.lakeflow.all_events`)
and before `Refund_Evaluation` (which reads from this dataset).

In [ ]:
%pip install -U -qqqq mlflow-skinny[databricks]
dbutils.library.restartPython()

In [ ]:
CATALOG = dbutils.widgets.get("CATALOG")
print(f"CATALOG = {CATALOG}")

In [ ]:
refund_queries = [
    row["order_id"]
    for row in spark.sql(
        f"""
        SELECT order_id
        FROM {CATALOG}.lakeflow.all_events
        WHERE event_type='delivered'
        LIMIT 10
        """
    ).collect()
]

EVAL_DATASET = [
    {
        "inputs": {
            "messages": [{"role": "user", "content": query}],
        }
    }
    for query in refund_queries
]

print(f"Built {len(EVAL_DATASET)} eval rows from {CATALOG}.lakeflow.all_events")

In [ ]:
import json as _json
import mlflow
import mlflow.genai.datasets

UC_DATASET_TABLE = f"{CATALOG}.evaluations.refund_agent_eval_dataset"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.evaluations")

# Drop + recreate to avoid Arrow schema mismatches if a prior run wrote a
# different shape (mirrors evaluation.ipynb for the supervisor).
spark.sql(f"DROP TABLE IF EXISTS {UC_DATASET_TABLE}")

eval_dataset = mlflow.genai.datasets.create_dataset(uc_table_name=UC_DATASET_TABLE)


def _serialize_record(rec):
    """JSON-stringify list-of-primitives fields only (Arrow + mlflow.genai.datasets
    can't reliably serialize ARRAY<STRING>). Lists of dicts (chat-format
    `inputs.messages`) MUST stay native — mlflow.genai.datasets validates them
    against the chat schema and fails with `AttributeError: 'str' object has no
    attribute 'get'` if they arrive as a stringified JSON. Eval-side reader
    deserializes the primitive lists back symmetrically."""
    out = {}
    for k, v in rec.items():
        if isinstance(v, dict):
            out[k] = _serialize_record(v)
        elif isinstance(v, list) and all(isinstance(x, (str, int, float, bool)) for x in v):
            out[k] = _json.dumps(v)
        else:
            out[k] = v
    return out


eval_dataset.merge_records([_serialize_record(r) for r in EVAL_DATASET])
print(f"✅ Registered {UC_DATASET_TABLE} ({len(EVAL_DATASET)} records)")

In [ ]:
dev_experiment_name = f"/Shared/{CATALOG}_refund_agent_dev"
mlflow.set_experiment(dev_experiment_name)

with mlflow.start_run(run_name="register_eval_dataset"):
    mlflow.log_input(
        mlflow.data.from_spark(
            spark.table(UC_DATASET_TABLE),
            table_name=UC_DATASET_TABLE,
        ),
        context="eval",
    )

print(f"✅ Linked {UC_DATASET_TABLE} to experiment {dev_experiment_name}")
print(f"   Use in evaluate(): mlflow.genai.datasets.get_dataset(uc_table_name=UC_DATASET_TABLE)")